In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("adham7elmy/deepfake-detection-dataset")

print("Path to dataset files:", path)

100%|██████████| 634M/634M [00:10<00:00, 66.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/adham7elmy/deepfake-detection-dataset/versions/1


In [ ]:
!pip install mtcnn opencv-python kagglehub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 92.2 MB/s eta 0:00:00


In [ ]:
!pip install lz4
!pip install --force-reinstall mtcnn


  Using cached mtcnn-1.0.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached lz4-4.4.5-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (3.8 kB)
Using cached mtcnn-1.0.0-py3-none-any.whl (1.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 12.7 MB/s eta 0:00:00
Using cached lz4-4.4.5-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (1.4 MB)
  Attempting uninstall: lz4
    Found existing installation: lz4 4.4.5
    Uninstalling lz4-4.4.5:
      Successfully uninstalled lz4-4.4.5
  Attempting uninstall: joblib
    Found existing installation: joblib 1.5.3
    Uninstalling joblib-1.5.3:
      Successfully uninstalled joblib-1.5.3
  Attempting uninstall: mtcnn
    Found existing installation: mtcnn 1.0.0
    Uninstalling mtcnn-1.0.0:
      Successfully uninstalled mtcnn-1.0.0


In [ ]:
from mtcnn import MTCNN
detector = MTCNN()
print("MTCNN working!")


MTCNN working!


In [ ]:
!pip install retina-face


In [ ]:
import cv2
import os
from retinaface import RetinaFace
from tqdm import tqdm

def extract_face(img_path, save_path):
    try:
        img = cv2.imread(img_path)
        faces = RetinaFace.detect_faces(img_path)

        if len(faces) > 0:
            face_info = faces[list(faces.keys())[0]]
            x1, y1, x2, y2 = face_info["facial_area"]

            face = img[y1:y2, x1:x2]
            face = cv2.resize(face, (224,224))

            cv2.imwrite(save_path, face)
            return True
        return False

    except:
        return False


In [ ]:
import tensorflow as tf
print("TF Version:", tf.__version__)


TF Version: 2.19.0


In [ ]:
!pip install retina-face
!pip install opencv-python


In [ ]:
import os
import shutil
import random
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model


In [ ]:
import kagglehub

path = kagglehub.dataset_download("adham7elmy/deepfake-detection-dataset")

print(path)

100%|██████████| 634M/634M [00:07<00:00, 89.4MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/adham7elmy/deepfake-detection-dataset/versions/1


In [ ]:
import shutil

src = "/root/.cache/kagglehub/datasets/adham7elmy/deepfake-detection-dataset/versions/1/dataset"
dst = "/content/deepfake_dataset"

shutil.copytree(src, dst)

print("Copied successfully")

Copied successfully


In [ ]:
!ls /content/deepfake_dataset

fake  real


In [ ]:
!find /content -type d | grep dataset

/content/deepfake_dataset
/content/deepfake_dataset/fake
/content/deepfake_dataset/real


In [ ]:
import os

base = "/content/deepfake_dataset"

real = len(os.listdir(base + "/real"))
fake = len(os.listdir(base + "/fake"))

print("Real images:", real)
print("Fake images:", fake)

Real images: 5982
Fake images: 7806


In [ ]:
!rm -rf /content/balanced_dataset


In [ ]:
!rm -rf /content/balanced_dataset
!rm -rf /content/split_data


In [ ]:
import os
import shutil
import random

src = "/root/.cache/kagglehub/datasets/adham7elmy/deepfake-detection-dataset/versions/1/dataset"
dst = "/content/balanced_dataset"

os.makedirs(dst, exist_ok=True)
os.makedirs(dst + "/real", exist_ok=True)
os.makedirs(dst + "/fake", exist_ok=True)

real_images = os.listdir(src + "/real")
fake_images = os.listdir(src + "/fake")

print("Before Balancing:")
print("Real:", len(real_images))
print("Fake:", len(fake_images))

min_count = min(len(real_images), len(fake_images))

real_sample = random.sample(real_images, min_count)
fake_sample = random.sample(fake_images, min_count)

for img in real_sample:
    shutil.copy(os.path.join(src, "real", img), os.path.join(dst, "real", img))

for img in fake_sample:
    shutil.copy(os.path.join(src, "fake", img), os.path.join(dst, "fake", img))

print("\nAfter Balancing:")
print("Real:", len(os.listdir(dst + "/real")))
print("Fake:", len(os.listdir(dst + "/fake")))


Before Balancing:
Real: 5982
Fake: 7806

After Balancing:
Real: 5982
Fake: 5982


In [ ]:
base = "/content/balanced_dataset"

split_dir = "/content/split_data"
os.makedirs(split_dir, exist_ok=True)

for t in ["train", "val", "test"]:
    for cls in ["real", "fake"]:
        os.makedirs(f"{split_dir}/{t}/{cls}", exist_ok=True)

for cls in ["real", "fake"]:
    images = os.listdir(f"{base}/{cls}")
    random.shuffle(images)

    n = len(images)

    train_split = int(0.7 * n)
    val_split = int(0.85 * n)

    train_imgs = images[:train_split]
    val_imgs = images[train_split:val_split]
    test_imgs = images[val_split:]

    for img in train_imgs:
        shutil.copy(f"{base}/{cls}/{img}", f"{split_dir}/train/{cls}/{img}")

    for img in val_imgs:
        shutil.copy(f"{base}/{cls}/{img}", f"{split_dir}/val/{cls}/{img}")

    for img in test_imgs:
        shutil.copy(f"{base}/{cls}/{img}", f"{split_dir}/test/{cls}/{img}")

print("Split Completed!")


Split Completed!


In [ ]:
print("Train Real:", len(os.listdir("/content/split_data/train/real")))
print("Train Fake:", len(os.listdir("/content/split_data/train/fake")))

print("Val Real:", len(os.listdir("/content/split_data/val/real")))
print("Val Fake:", len(os.listdir("/content/split_data/val/fake")))

print("Test Real:", len(os.listdir("/content/split_data/test/real")))
print("Test Fake:", len(os.listdir("/content/split_data/test/fake")))


Train Real: 4187
Train Fake: 4187
Val Real: 897
Val Fake: 897
Test Real: 898
Test Fake: 898


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.25,
    brightness_range=[0.7,1.3],
    horizontal_flip=True
)

test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    "/content/split_data/train",
    target_size=(224,224),
    batch_size=16,
    class_mode='binary'
)

val_data = test_gen.flow_from_directory(
    "/content/split_data/val",
    target_size=(224,224),
    batch_size=16,
    class_mode='binary'
)

test_data = test_gen.flow_from_directory(
    "/content/split_data/test",
    target_size=(224,224),
    batch_size=16,
    class_mode='binary'
)


Found 8374 images belonging to 2 classes.
Found 1794 images belonging to 2 classes.
Found 1796 images belonging to 2 classes.


In [ ]:
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
import tensorflow as tf

base = Xception(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

for layer in base.layers:
    layer.trainable = False

x = base.output
x = GlobalAveragePooling2D()(x)

x = Dense(512, activation='relu')(x)
x = Dropout(0.6)(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

out = Dense(1, activation='sigmoid')(x)

model = Model(base.input, out)


In [ ]:
callbacks = [

    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc',
        patience=5,
        mode='max',
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-7
    ),

    tf.keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor="val_accuracy",
        save_best_only=True
    )

]


In [ ]:
import tensorflow as tf
print("GPU Available:", tf.config.list_physical_devices('GPU'))

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-5),
    loss='binary_crossentropy',
    metrics=['accuracy','auc']
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=8,
    callbacks=callbacks
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/8
524/524 ━━━━━━━━━━━━━━━━━━━━ 195s 330ms/step - accuracy: 0.5936 - auc: 0.6187 - loss: 0.6722 - val_accuracy: 0.7096 - val_auc: 0.7949 - val_loss: 0.5468 - learning_rate: 5.0000e-05
Epoch 2/8
524/524 ━━━━━━━━━━━━━━━━━━━━ 141s 270ms/step - accuracy: 0.6944 - auc: 0.7670 - loss: 0.5727 - val_accuracy: 0.7263 - val_auc: 0.8156 - val_loss: 0.5152 - learning_rate: 5.0000e-05
Epoch 3/8
524/524 ━━━━━━━━━━━━━━━━━━━━ 141s 269ms/step - accuracy: 0.7170 - auc: 0.7890 - loss: 0.5492 - val_accuracy: 0.7302 - val_auc: 0.8229 - val_loss: 0.5035 - learning_rate: 5.0000e-05
Epoch 4/8
524/524 ━━━━━━━━━━━━━━━━━━━━ 141s 268ms/step - accuracy: 0.7321 - auc: 0.8070 - loss: 0.5235 - val_accuracy: 0.7358 - val_auc: 0.8307 - val_loss: 0.4949 - learning_rate: 5.0000e-05
Epoch 5/8
524/524 ━━━━━━━━━━━━━━━━━━━━ 142s 270ms/step - accuracy: 0.7415 - auc: 0.8221 - loss: 0.5046 - val_accuracy: 0.7436 - val_auc: 0.8391 - val_loss: 0.4767 - learning_rate: 5.0000e-05
Epoch 6/8
524/524 ━━━━━━━━━━━━━━━━━━━━ 140s 2

In [ ]:
for layer in base.layers[-60:]:
    layer.trainable = True
#UNFREEZING LAST LAYERS

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy','auc']
)
#RECOMPILING WITH SMALLER LR

In [ ]:
history2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks
)


Epoch 1/10
524/524 ━━━━━━━━━━━━━━━━━━━━ 196s 324ms/step - accuracy: 0.7340 - auc: 0.8085 - loss: 0.5296 - val_accuracy: 0.7832 - val_auc: 0.8841 - val_loss: 0.4148 - learning_rate: 1.0000e-05
Epoch 2/10
524/524 ━━━━━━━━━━━━━━━━━━━━ 149s 285ms/step - accuracy: 0.7857 - auc: 0.8758 - loss: 0.4305 - val_accuracy: 0.8255 - val_auc: 0.9239 - val_loss: 0.3402 - learning_rate: 1.0000e-05
Epoch 3/10
524/524 ━━━━━━━━━━━━━━━━━━━━ 149s 284ms/step - accuracy: 0.8311 - auc: 0.9160 - loss: 0.3648 - val_accuracy: 0.8623 - val_auc: 0.9456 - val_loss: 0.2913 - learning_rate: 1.0000e-05
Epoch 4/10
524/524 ━━━━━━━━━━━━━━━━━━━━ 152s 289ms/step - accuracy: 0.8578 - auc: 0.9356 - loss: 0.3216 - val_accuracy: 0.8802 - val_auc: 0.9603 - val_loss: 0.2542 - learning_rate: 1.0000e-05
Epoch 5/10
524/524 ━━━━━━━━━━━━━━━━━━━━ 149s 283ms/step - accuracy: 0.8773 - auc: 0.9537 - loss: 0.2745 - val_accuracy: 0.9030 - val_auc: 0.9686 - val_loss: 0.2317 - learning_rate: 1.0000e-05
Epoch 6/10
524/524 ━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    filepath="/content/drive/MyDrive/deepfake_best.h5",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)


In [ ]:
history2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=3,
    callbacks=[checkpoint]
)


Epoch 1/3
524/524 ━━━━━━━━━━━━━━━━━━━━ 0s 277ms/step - accuracy: 0.9405 - auc: 0.9849 - loss: 0.1571
Epoch 1: val_accuracy improved from -inf to 0.95429, saving model to /content/drive/MyDrive/deepfake_best.h5


524/524 ━━━━━━━━━━━━━━━━━━━━ 161s 307ms/step - accuracy: 0.9405 - auc: 0.9849 - loss: 0.1571 - val_accuracy: 0.9543 - val_auc: 0.9886 - val_loss: 0.1308
Epoch 2/3
524/524 ━━━━━━━━━━━━━━━━━━━━ 0s 279ms/step - accuracy: 0.9412 - auc: 0.9871 - loss: 0.1447
Epoch 2: val_accuracy improved from 0.95429 to 0.95819, saving model to /content/drive/MyDrive/deepfake_best.h5


524/524 ━━━━━━━━━━━━━━━━━━━━ 156s 298ms/step - accuracy: 0.9412 - auc: 0.9871 - loss: 0.1447 - val_accuracy: 0.9582 - val_auc: 0.9907 - val_loss: 0.1152
Epoch 3/3
524/524 ━━━━━━━━━━━━━━━━━━━━ 0s 278ms/step - accuracy: 0.9533 - auc: 0.9903 - loss: 0.1250
Epoch 3: val_accuracy did not improve from 0.95819
524/524 ━━━━━━━━━━━━━━━━━━━━ 153s 292ms/step - accuracy: 0.9533 - auc: 0.9903 - loss: 0.1250 - val_accuracy: 0.9582 - val_auc: 0.9899 - val_loss: 0.1214


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from tensorflow.keras.models import load_model

model = load_model("/content/drive/MyDrive/deepfake_detector_final.h5")

In [ ]:
print(model.summary())

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 111, 111,  │        864 │ input_layer[0][0] │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_bn     │ (None, 111, 111,  │        128 │ block1_conv1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_act    │ (None, 111, 111,  │          0 │ block1_conv1_bn[… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 109, 109,  │     18,432 │ block1_conv1_act… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_bn     │ (None, 109, 109,  │        256 │ block1_conv2[0][… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_act    │ (None, 109, 109,  │          0 │ block1_conv2_bn[… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1     │ (None, 109, 109,  │      8,768 │ block1_conv2_act… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1_bn  │ (None, 109, 109,  │        512 │ block2_sepconv1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_act │ (None, 109, 109,  │          0 │ block2_sepconv1_… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2     │ (None, 109, 109,  │     17,536 │ block2_sepconv2_… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_bn  │ (None, 109, 109,  │        512 │ block2_sepconv2[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 55, 55,    │      8,192 │ block1_conv2_act… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 55, 55,    │          0 │ block2_sepconv2_… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 55, 55,    │        512 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 55, 55,    │          0 │ block2_pool[0][0… │
│                     │ 128)              │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_sepconv1_act │ (None, 55, 55,    │          0 │ add[0][0]       

 Total params: 22,042,155 (84.08 MB)

 Trainable params: 14,962,953 (57.08 MB)

 Non-trainable params: 7,079,200 (27.01 MB)

 Optimizer params: 2 (12.00 B)

None


In [ ]:
print(test_data.class_indices)

{'fake': 0, 'real': 1}


In [ ]:
!ls /content/drive/MyDrive | grep deepfake

deepfake_best.h5
deepfake_detector_final.h5
deepfake_detector_final.keras


In [ ]:
from google.colab import files
from tensorflow.keras.preprocessing import image
import numpy as np

uploaded = files.upload()

filename = list(uploaded.keys())[0]

img = image.load_img(filename, target_size=(224,224))

img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

prediction = model.predict(img_array)

print("Raw Output:", prediction)

if prediction[0][0] > 0.5:
    print("REAL IMAGE")
else:
    print("FAKE IMAGE")

Saving 3969b4033a5a5b2f7204876edae50ff4.gif to 3969b4033a5a5b2f7204876edae50ff4.gif
1/1 ━━━━━━━━━━━━━━━━━━━━ 15s 15s/step
Raw Output: [[0.99893886]]
REAL IMAGE


In [ ]:
prediction = model.predict(img_array)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step


In [ ]:

%%writefile deepfake-detection-system/app.py


Writing deepfake-detection-system/app.py


In [ ]:
!ls deepfake-detection-system


app.py	model


In [ ]:
%%writefile deepfake-detection-system/app.py
!pip install gradio
from tensorflow.keras.models import load_model
import numpy as np
import cv2

model = load_model("/content/drive/MyDrive/deepfake_detector_final.keras")
print(train_data.class_indices)
def predict_image(image):
    image = np.array(image)

    # Handle RGBA images
    if image.shape[-1] == 4:
        image = image[:, :, :3]

    image = cv2.resize(image, (224, 224))
    image = image / 255.0
    image = np.expand_dims(image, axis=0)

    prediction = model.predict(image)[0][0]

    if prediction >= 0.5:
        label = "REAL"
        confidence = prediction
    else:
        label = "FAKE"
        confidence = 1 - prediction

    return f"{label} (Confidence: {confidence:.2f})"
import gradio as gr

interface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil", label="Upload Image"),
    outputs=gr.Textbox(label="Prediction"),
    title="Deepfake Detection System",
    description="Upload an image to check whether it is REAL or FAKE."
)

interface.launch()



Overwriting deepfake-detection-system/app.py


In [ ]:
!ls deepfake-detection-system


app.py	model


In [ ]:
%%writefile deepfake-detection-system/requirements.txt
tensorflow
numpy
opencv-python
gradio


Writing deepfake-detection-system/requirements.txt


In [ ]:
%%writefile deepfake-detection-system/README.md
# Deepfake Detection System

A deep learning–based system to detect whether an image is REAL or FAKE.

## Features
- CNN-based deepfake detection
- Binary classification (Real vs Fake)
- Confidence score
- Gradio UI

## Tech Stack
- Python
- TensorFlow / Keras
- OpenCV
- Gradio

## How to Run
pip install -r requirements.txt
python app.py

## Model
The trained model is not included due to size limits.


Writing deepfake-detection-system/README.md


In [ ]:
!ls deepfake-detection-system


app.py	model  README.md  requirements.txt


In [ ]:
!zip -r deepfake-detection-system.zip deepfake-detection-system


  adding: deepfake-detection-system/ (stored 0%)
  adding: deepfake-detection-system/requirements.txt (stored 0%)
  adding: deepfake-detection-system/model/ (stored 0%)
  adding: deepfake-detection-system/README.md (deflated 31%)
  adding: deepfake-detection-system/app.py (deflated 48%)


In [ ]:
from google.colab import files
files.download("deepfake-detection-system.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>